In [3]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')          # change to 'TkAgg' if you want a popup window
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [4]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.preprocessing  import StandardScaler
from sklearn.linear_model     import LinearRegression
from sklearn.metrics        import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

In [6]:
print("Loading data...")
df = pd.read_csv('D:\\Users\\Md Mahfuzur Rahman\\Desktop\\Research\\for_me\\cleaned_combined_data.csv')

df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values('datetime').reset_index(drop=True)


Loading data...


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 42611 entries, 0 to 42610
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   datetime       42611 non-null  datetime64[us]
 1   demand_mw      42611 non-null  int64         
 2   load_shedding  42611 non-null  int64         
 3   temp_mean      42611 non-null  float64       
 4   wspd_mean      42611 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(2)
memory usage: 1.6 MB


In [5]:
df.head(5)

,datetime,demand_mw,load_shedding,temp_mean,wspd_mean
0,2020-01-01 03:00:00,5351,0,18.65,1.80
1,2020-01-01 06:00:00,5346,0,23.73,3.00
2,2020-01-01 09:00:00,6654,0,25.25,4.58
3,2020-01-01 12:00:00,6750,0,21.05,2.17
4,2020-01-01 15:00:00,6546,0,18.82,0.30


In [6]:
df['hour']       = df['datetime'].dt.hour
df['dayofweek']  = df['datetime'].dt.dayofweek
df['month']      = df['datetime'].dt.month
df['dayofyear']  = df['datetime'].dt.dayofyear
df['lag1']       = df['demand_mw'].shift(1)
df['lag24']      = df['demand_mw'].shift(24)
df['lag168']     = df['demand_mw'].shift(168)
df['roll24_mean']= df['demand_mw'].shift(1).rolling(24).mean()
df['roll24_std'] = df['demand_mw'].shift(1).rolling(24).std()
df               = df.dropna().reset_index(drop=True)

FEATURES = ['hour','dayofweek','month','dayofyear','load_shedding','temp_mean','wspd_mean',
            'lag1','lag24','lag168','roll24_mean','roll24_std']
TARGET   = 'demand_mw'

In [7]:
df.head()

,datetime,demand_mw,load_shedding,temp_mean,wspd_mean,hour,dayofweek,month,dayofyear,lag1,lag24,lag168,roll24_mean,roll24_std
0,2020-03-08 12:00:00,7960,0,26.27,2.17,12,6,3,68,7532.0,5947.0,5351.0,7169.291667,973.876721
1,2020-03-08 15:00:00,7768,0,23.12,2.77,15,6,3,68,7960.0,6156.0,5346.0,7253.166667,950.432564
2,2020-03-08 18:00:00,7721,0,21.47,1.80,18,6,3,68,7768.0,7672.0,6654.0,7320.333333,926.175242
3,2020-03-08 21:00:00,9241,0,20.03,1.20,21,6,3,68,7721.0,8138.0,6750.0,7322.375000,927.037770
4,2020-03-09 03:00:00,6511,0,23.52,6.72,3,0,3,69,9241.0,8172.0,6546.0,7368.333333,994.142920


In [8]:
from sklearn.model_selection import train_test_split
X = df[FEATURES]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Reset indices for easier access
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

# Create aliases for backward compatibility
X_tr, X_te = X_train, X_test
y_tr, y_te = y_train, y_test


In [10]:
# ── 2. TRAIN / TEST SPLIT (80 / 20) ──────────────────────────────────────────
import os
OUTPUT_DIR = '.'  # Save results to current directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

sc        = StandardScaler()
X_tr_sc   = sc.fit_transform(X_train)
X_te_sc   = sc.transform(X_test)



print(f"Train rows: {len(X_train):,}   Test rows: {len(X_test):,}")


Train rows: 33,954   Test rows: 8,489


In [25]:
# ── 3. METRIC HELPER ─────────────────────────────────────────────────────────
def metrics(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2   = r2_score(y_true, y_pred)
    acc  = max(0.0, 1.0 - mape / 100.0) * 100
    return dict(Model=name,
                Accuracy=round(acc, 3),
                MAE=round(mae, 1),
                RMSE=round(rmse, 1),
                MAPE=round(mape, 3),
                R2=round(r2, 4))


In [26]:
results = []

# ─────────────────────────────────────────────────────────────────────────────
#  SECTION A — STATISTICAL / HYBRID MODELS
# ─────────────────────────────────────────────────────────────────────────────

# ── A1. ARIMA (AR lag regression proxy) ──────────────────────────────────────
print("Running ARIMA...")
AR_FEAT = ['lag1', 'lag24', 'lag168', 'roll24_mean']
m_arima = LinearRegression()
m_arima.fit(X_tr_sc[:, [FEATURES.index(f) for f in AR_FEAT]], y_train)
pred_arima = m_arima.predict(X_te_sc[:, [FEATURES.index(f) for f in AR_FEAT]])
results.append(metrics('ARIMA', y_test, pred_arima))




Running ARIMA...


In [27]:
# ── A2. SARIMA (AR + seasonal dummies) ───────────────────────────────────────
print("Running SARIMA...")
S_FEAT = ['lag1','lag24','lag168','roll24_mean','hour','month','dayofweek']
m_sarima = LinearRegression()
m_sarima.fit(X_tr_sc[:, [FEATURES.index(f) for f in S_FEAT]], y_train)
pred_sarima = m_sarima.predict(X_te_sc[:, [FEATURES.index(f) for f in S_FEAT]])
results.append(metrics('SARIMA', y_test, pred_sarima))

Running SARIMA...


In [28]:
# ── B2. Gradient Boosting ─────────────────────────────────────────────────────
print("Running Gradient Boosting...")
m_gb = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05,
                                  max_depth=5, random_state=42)
m_gb.fit(X_tr_sc, y_tr)
pred_gb = m_gb.predict(X_te_sc)
results.append(metrics('Gradient Boosting', y_te, pred_gb))

Running Gradient Boosting...


In [29]:
# ── B3. XGBoost (sklearn GradientBoosting with XGB-style params) ─────────────
import xgboost
print("Running XGBoost...")
m_xgb = xgboost.XGBRegressor(n_estimators=300, learning_rate=0.03,
                              max_depth=5, subsample=0.8,
                              min_samples_leaf=10, random_state=42)
m_xgb.fit(X_tr_sc, y_tr)
pred_xgb = m_xgb.predict(X_te_sc)
results.append(metrics('XGBoost', y_te, pred_xgb))


Running XGBoost...


In [30]:
#  SECTION C — DEEP LEARNING MODELS  (numpy proxies)

# ── C1. TCN (Temporal Convolutional Network) — multi-scale rolling features ──
print("Running TCN...")
TCN_SCALES = [1, 2, 4, 8, 16, 24, 48]
tcn_tr = [pd.Series(y_tr).shift(1).rolling(s).mean().bfill().values
          for s in TCN_SCALES]
tcn_te = []
for s in TCN_SCALES:
    combined = np.concatenate([y_tr[-60:], y_te])
    rolled   = pd.Series(combined).shift(1).rolling(s).mean().bfill().values
    tcn_te.append(rolled[60:])
X_tcn_tr = np.column_stack(tcn_tr + [X_tr_sc])
X_tcn_te = np.column_stack(tcn_te + [X_te_sc])
m_tcn = RandomForestRegressor(n_estimators=80, max_depth=10,
                               n_jobs=-1, random_state=99)
m_tcn.fit(X_tcn_tr, y_tr)
pred_tcn = m_tcn.predict(X_tcn_te)
results.append(metrics('TCN', y_te, pred_tcn))

Running TCN...


In [31]:
# ── C2. Transformer — multi-head attention proxy via lag ensemble + GB ────────
print("Running Transformer...")
ATTN_LAGS = [1, 2, 3, 6, 12, 24, 48, 72, 168]
lag_tr = np.column_stack([pd.Series(y_tr).shift(l).bfill().values
                           for l in ATTN_LAGS])
lag_te = []
for l in ATTN_LAGS:
    c = np.concatenate([y_tr[-200:], y_te])
    lag_te.append(pd.Series(c).shift(l).bfill().values[200:])
lag_te = np.column_stack(lag_te)
X_trans_tr = np.hstack([lag_tr, X_tr_sc])
X_trans_te = np.hstack([lag_te, X_te_sc])
m_trans = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05,
                                     max_depth=4, subsample=0.8, random_state=7)
m_trans.fit(X_trans_tr, y_tr)
pred_trans = m_trans.predict(X_trans_te)
results.append(metrics('Transformer', y_te, pred_trans))

Running Transformer...


In [32]:
# ── D3. Conformal Prediction (split conformal, 90 % coverage target) ─────────
print("Running Conformal Prediction...")
CALIB_FRAC  = 0.20
calib_n     = int(CALIB_FRAC * len(X_tr_sc))
X_tr2, y_tr2 = X_tr_sc[:-calib_n], y_tr[:-calib_n]
X_cal, y_cal = X_tr_sc[-calib_n:],  y_tr[-calib_n:]
m_cp = RandomForestRegressor(n_estimators=80, max_depth=10,
                              n_jobs=-1, random_state=0)
m_cp.fit(X_tr2, y_tr2)
residuals   = np.abs(y_cal - m_cp.predict(X_cal))
q_conf      = np.quantile(residuals, 0.90)          # 90 % coverage
pred_cp     = m_cp.predict(X_te_sc)
cp_ci_lo    = pred_cp - q_conf
cp_ci_hi    = pred_cp + q_conf
cp_coverage = np.mean((y_te >= cp_ci_lo) & (y_te <= cp_ci_hi)) * 100
results.append({**metrics('Conformal Prediction', y_te, pred_cp),
                'CI_Coverage_90pct': round(cp_coverage, 1)})


Running Conformal Prediction...


In [33]:
# ── 4. RESULTS TABLE ─────────────────────────────────────────────────────────
df_res = pd.DataFrame(results).sort_values('Accuracy', ascending=False)
df_res.insert(0, 'Rank', range(1, len(df_res) + 1))
df_res['CI_Coverage_90pct'] = df_res.get('CI_Coverage_90pct', np.nan)

print("\n" + "="*80)
print("  ELECTRICITY DEMAND FORECASTING — ALL 15 MODEL RESULTS")
print("="*80)
print(df_res[['Rank','Model','Accuracy','MAE','RMSE','MAPE','R2',
              'CI_Coverage_90pct']].to_string(index=False))
print("="*80)

df_res.to_csv(f'{OUTPUT_DIR}/model_results_top_8.csv', index=False)
print(f"\nResults saved → {OUTPUT_DIR}/model_results_top_8.csv")



  ELECTRICITY DEMAND FORECASTING — ALL 15 MODEL RESULTS
 Rank                Model  Accuracy   MAE  RMSE  MAPE     R2  CI_Coverage_90pct
    1    Gradient Boosting    97.994 233.7 309.7 2.006 0.9806                NaN
    2              XGBoost    97.980 236.0 313.0 2.020 0.9802                NaN
    3                  TCN    97.901 241.1 339.9 2.099 0.9766                NaN
    4          Transformer    97.793 258.4 341.6 2.207 0.9764                NaN
    5 Conformal Prediction    97.602 280.9 391.7 2.398 0.9689               89.3
    6               SARIMA    96.967 343.4 450.9 3.033 0.9589                NaN
    7                ARIMA    96.743 369.0 495.5 3.257 0.9503                NaN

Results saved → ./model_results_top_8.csv


In [34]:
# ── 5. VISUALIZATION ─────────────────────────────────────────────────────────
CAT_COLOR = {
    'ARIMA':               '#4E79A7',
    'SARIMA':              '#4E79A7',
    'Gradient Boosting':   '#F28E2B',
    'XGBoost':             '#F28E2B',
    'LSTM':                '#59A14F',
    'TCN':                 '#59A14F',
    'Transformer':         '#59A14F',
}

models_sorted  = df_res['Model'].tolist()
accuracy_sorted= df_res['Accuracy'].tolist()
mae_sorted     = df_res['MAE'].tolist()
rmse_sorted    = df_res['RMSE'].tolist()
mape_sorted    = df_res['MAPE'].tolist()
r2_sorted      = df_res['R2'].tolist()
colors_sorted  = [CAT_COLOR.get(m, '#888') for m in models_sorted]

fig, axes = plt.subplots(3, 2, figsize=(20, 24), facecolor='#0F1923')
fig.patch.set_facecolor('#0F1923')


In [35]:
def style_ax(ax, title, xlabel):
    ax.set_facecolor('#151F2B')
    ax.set_title(title, color='white', fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel(xlabel, color='#AABBCC', fontsize=10)
    ax.tick_params(colors='#AABBCC', labelsize=8.5)
    ax.xaxis.grid(True, color='#223344', linestyle='--', alpha=0.5)
    ax.set_axisbelow(True)
    for sp in ax.spines.values():
        sp.set_visible(False)


In [36]:
# ── Plot 1: Accuracy (spans full top row) ────────────────────────────────────
ax1 = plt.subplot2grid((3, 2), (0, 0), colspan=2, fig=fig)
ax1.set_facecolor('#151F2B')
bars = ax1.barh(models_sorted[::-1], accuracy_sorted[::-1],
                color=colors_sorted[::-1], edgecolor='none', height=0.65)
for i, bar in enumerate(bars):
    idx = len(models_sorted) - 1 - i
    val = accuracy_sorted[idx]
    if val >= 99.9:
        bar.set_edgecolor('gold'); bar.set_linewidth(1.5)
    ax1.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
             f'{val:.3f}%', va='center', color='white', fontsize=8.5)
ax1.set_xlim(82, 102)
style_ax(ax1, 'Model Accuracy — All 8 Models (sorted best → worst)', 'Accuracy (%)')
legend_patches = [
    mpatches.Patch(color='#4E79A7', label='Statistical / Hybrid'),
    mpatches.Patch(color='#F28E2B', label='Machine Learning'),
    mpatches.Patch(color='#59A14F', label='Deep Learning'),
    mpatches.Patch(color='#B07AA1', label='Probabilistic'),
]
ax1.legend(handles=legend_patches, loc='lower right',
           framealpha=0.25, labelcolor='white',
           facecolor='#223344', fontsize=9)


In [37]:
# ── Plot 2: MAE ───────────────────────────────────────────────────────────────
ax2 = plt.subplot2grid((3, 2), (1, 0), fig=fig)
ax2.barh(models_sorted[::-1], mae_sorted[::-1],
         color=colors_sorted[::-1], edgecolor='none', height=0.65)
style_ax(ax2, 'MAE — Mean Absolute Error (lower = better)', 'MAE (MW)')

In [38]:
# ── Plot 3: RMSE ──────────────────────────────────────────────────────────────
ax3 = plt.subplot2grid((3, 2), (1, 1), fig=fig)
ax3.barh(models_sorted[::-1], rmse_sorted[::-1],
         color=colors_sorted[::-1], edgecolor='none', height=0.65)
style_ax(ax3, 'RMSE — Root Mean Squared Error (lower = better)', 'RMSE (MW)')

In [39]:
# ── Plot 4: R² ────────────────────────────────────────────────────────────────
ax4 = plt.subplot2grid((3, 2), (2, 0), fig=fig)
ax4.barh(models_sorted[::-1], r2_sorted[::-1],
         color=colors_sorted[::-1], edgecolor='none', height=0.65)
style_ax(ax4, 'R² Score (higher = better)', 'R²')

In [40]:
# ── Plot 5: MAPE dot chart ────────────────────────────────────────────────────
ax5 = plt.subplot2grid((3, 2), (2, 1), fig=fig)
ax5.set_facecolor('#151F2B')
y_pos = range(len(models_sorted))
ax5.scatter(mape_sorted[::-1], list(y_pos),
            c=colors_sorted[::-1], s=90, zorder=3)
ax5.set_yticks(list(y_pos))
ax5.set_yticklabels(models_sorted[::-1], fontsize=8.5)
style_ax(ax5, 'MAPE — Mean Absolute Percentage Error (lower = better)', 'MAPE (%)')

fig.suptitle('Electricity Demand Forecasting — Hybrid Framework\nAll 8 Model Performance Results',
             color='white', fontsize=16, fontweight='bold', y=0.98)

chart_path = f'{OUTPUT_DIR}/model_results_top_8.png'
plt.savefig(chart_path, dpi=150, bbox_inches='tight', facecolor='#0F1923')
print(f"Chart saved  → {chart_path}")
plt.close()

print("\nDone! Top 5 models:")
print(df_res[['Rank','Model','Accuracy','MAE','RMSE','R2']].head(5).to_string(index=False))

Chart saved  → ./model_results_top_8.png

Done! Top 5 models:
 Rank                Model  Accuracy   MAE  RMSE     R2
    1    Gradient Boosting    97.994 233.7 309.7 0.9806
    2              XGBoost    97.980 236.0 313.0 0.9802
    3                  TCN    97.901 241.1 339.9 0.9766
    4          Transformer    97.793 258.4 341.6 0.9764
    5 Conformal Prediction    97.602 280.9 391.7 0.9689


In [41]:
# ── 6. FEATURE IMPORTANCE ANALYSIS ──────────────────────────────────────────

print("\n" + "="*80)
print("  FEATURE IMPORTANCE ANALYSIS FOR ALL MODELS")
print("="*80)

# Dictionary to store feature importance for each model
feature_importance_dict = {}

# ── Tree-based models with feature_importances_ ──────────────────────────────
# Gradient Boosting
if hasattr(m_gb, 'feature_importances_'):
    feature_importance_dict['Gradient Boosting'] = m_gb.feature_importances_
    print("✓ Gradient Boosting feature importance extracted")

# XGBoost
if hasattr(m_xgb, 'feature_importances_'):
    feature_importance_dict['XGBoost'] = m_xgb.feature_importances_
    print("✓ XGBoost feature importance extracted")

# TCN (Random Forest)
if hasattr(m_tcn, 'feature_importances_'):
    feature_importance_dict['TCN'] = m_tcn.feature_importances_
    print("✓ TCN feature importance extracted")

# Transformer (Gradient Boosting)
if hasattr(m_trans, 'feature_importances_'):
    feature_importance_dict['Transformer'] = m_trans.feature_importances_
    print("✓ Transformer feature importance extracted")

# Conformal Prediction (Random Forest)
if hasattr(m_cp, 'feature_importances_'):
    feature_importance_dict['Conformal Prediction'] = m_cp.feature_importances_
    print("✓ Conformal Prediction feature importance extracted")

# ── Linear models - use absolute coefficients ───────────────────────────────
# ARIMA (Linear Regression on AR features)
if hasattr(m_arima, 'coef_'):
    arima_importance = np.abs(m_arima.coef_)
    feature_importance_dict['ARIMA'] = arima_importance
    print("✓ ARIMA coefficients importance extracted")

# SARIMA (Linear Regression)
if hasattr(m_sarima, 'coef_'):
    sarima_importance = np.abs(m_sarima.coef_)
    feature_importance_dict['SARIMA'] = sarima_importance
    print("✓ SARIMA coefficients importance extracted")

print("\n" + "="*80)


  FEATURE IMPORTANCE ANALYSIS FOR ALL MODELS
✓ Gradient Boosting feature importance extracted
✓ XGBoost feature importance extracted
✓ TCN feature importance extracted
✓ Transformer feature importance extracted
✓ Conformal Prediction feature importance extracted
✓ ARIMA coefficients importance extracted
✓ SARIMA coefficients importance extracted



In [42]:
# ── 6.1 VISUALIZATION: Individual Model Feature Importance ──────────────────

fig_fi, axes_fi = plt.subplots(4, 4, figsize=(24, 18), facecolor='#0F1923')
axes_fi = axes_fi.flatten()
fig_fi.patch.set_facecolor('#0F1923')

# Feature dimension mapping for models with different feature counts
feature_names_map = {
    'TCN': [f'rolling_scale_{i}' for i in range(7)] + FEATURES,  # 7 rolling scales + 12 features
    'Transformer': [f'attn_lag_{i}' for i in range(9)] + FEATURES,  # 9 lag features + 12 features
}

for idx, (model_name, importances) in enumerate(sorted(feature_importance_dict.items())):
    ax = axes_fi[idx]
    ax.set_facecolor('#151F2B')
    
    # Determine feature names for this model
    if model_name in feature_names_map:
        feat_names = feature_names_map[model_name][:len(importances)]
    else:
        feat_names = FEATURES[:len(importances)]
    
    # Sort by importance
    n_top = min(12, len(importances))
    sorted_idx = np.argsort(importances)[-n_top:]
    sorted_imp = importances[sorted_idx]
    sorted_feat = [feat_names[i] for i in sorted_idx]
    
    # Plot
    colors_fi = [CAT_COLOR.get(model_name, '#888')] * len(sorted_feat)
    bars_fi = ax.barh(sorted_feat, sorted_imp, color=colors_fi, edgecolor='white', linewidth=0.5)
    
    ax.set_title(f'{model_name} - Top {n_top} Features', color='white', fontsize=11, fontweight='bold', pad=8)
    ax.set_xlabel('Importance', color='#AABBCC', fontsize=9)
    ax.tick_params(colors='#AABBCC', labelsize=8)
    ax.xaxis.grid(True, color='#223344', linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)
    
    for spine in ax.spines.values():
        spine.set_visible(False)

# Hide extra subplots
for idx in range(len(feature_importance_dict), len(axes_fi)):
    axes_fi[idx].set_visible(False)

fig_fi.suptitle('Model Feature Importance - Individual Models\nShowing Top Features per Model',
                 color='white', fontsize=14, fontweight='bold', y=0.995)

fi_path = f'{OUTPUT_DIR}/feature_importance_by_model.png'
plt.savefig(fi_path, dpi=150, bbox_inches='tight', facecolor='#0F1923')
print(f"\nFeature importance chart saved → {fi_path}")
plt.close()

# ── 6.2 TABULAR FEATURE IMPORTANCE SUMMARY ──────────────────────────────────

print("\n" + "="*80)
print("  FEATURE IMPORTANCE RANKING BY MODEL (Top 5)")
print("="*80)

for model_name, importances in sorted(feature_importance_dict.items()):
    # Get feature names
    if model_name in feature_names_map:
        feat_names = feature_names_map[model_name][:len(importances)]
    else:
        feat_names = FEATURES[:len(importances)]
    
    sorted_idx = np.argsort(importances)[::-1]
    n_show = min(5, len(importances))
    top_features = [feat_names[i] for i in sorted_idx[:n_show]]
    top_importance = importances[sorted_idx[:n_show]]
    
    print(f"\n{model_name}:")
    for i, (feat, imp) in enumerate(zip(top_features, top_importance), 1):
        print(f"  {i}. {feat:25s} → {imp:8.4f}")



Feature importance chart saved → ./feature_importance_by_model.png

  FEATURE IMPORTANCE RANKING BY MODEL (Top 5)

ARIMA:
  1. hour                      → 2008.0860
  2. dayofyear                 → 332.5270
  3. month                     →  85.1113
  4. dayofweek                 →  45.0432

Conformal Prediction:
  1. lag1                      →   0.9366
  2. hour                      →   0.0386
  3. roll24_mean               →   0.0124
  4. temp_mean                 →   0.0046
  5. lag168                    →   0.0017

Gradient Boosting:
  1. lag1                      →   0.9443
  2. hour                      →   0.0355
  3. roll24_mean               →   0.0103
  4. temp_mean                 →   0.0034
  5. load_shedding             →   0.0018

SARIMA:
  1. hour                      → 1824.9449
  2. dayofyear                 → 575.5515
  3. load_shedding             → 293.3064
  4. dayofweek                 → 179.0752
  5. month                     →  39.5628

TCN:
  1. rolling_scale_

In [43]:
# ── 6.2 AGGREGATED FEATURE IMPORTANCE (Average across all models) ───────────

print("\n" + "="*80)
print("  AGGREGATED FEATURE IMPORTANCE (Average across all models)")
print("="*80)

# Only aggregate features that are common to all standard models (base 12 FEATURES)
aggregated_importance = {}

for model_name, importances in feature_importance_dict.items():
    # Extract only the base FEATURES if model has more
    if len(importances) >= len(FEATURES):
        base_imp = importances[:len(FEATURES)]
    else:
        base_imp = importances
    
    # Normalize to 0-1 scale
    imp_min = base_imp.min()
    imp_max = base_imp.max()
    if imp_max - imp_min > 0:
        normalized = (base_imp - imp_min) / (imp_max - imp_min)
    else:
        normalized = np.ones_like(base_imp) / len(base_imp)
    
    # Ensure it has 12 dimensions
    if len(normalized) < len(FEATURES):
        normalized = np.pad(normalized, (0, len(FEATURES) - len(normalized)), mode='constant')
    
    aggregated_importance[model_name] = normalized[:len(FEATURES)]

# Calculate average importance across all models (base features only)
avg_importance = np.mean(list(aggregated_importance.values()), axis=0)
feature_avg_df = pd.DataFrame({
    'Feature': FEATURES,
    'Avg_Importance': avg_importance
}).sort_values('Avg_Importance', ascending=False)

print("\n" + feature_avg_df.to_string(index=False))

# Visualization: Aggregated importance
fig_agg, ax_agg = plt.subplots(figsize=(14, 8), facecolor='#0F1923')
fig_agg.patch.set_facecolor('#0F1923')
ax_agg.set_facecolor('#151F2B')

sorted_features = feature_avg_df['Feature'].values
sorted_importance = feature_avg_df['Avg_Importance'].values
colors_agg = ['#FFD700' if imp > sorted_importance.mean() else '#4E79A7' 
              for imp in sorted_importance]

bars_agg = ax_agg.barh(sorted_features[::-1], sorted_importance[::-1], 
                        color=colors_agg[::-1], edgecolor='white', linewidth=1)

# Add value labels
for i, (bar, val) in enumerate(zip(bars_agg, sorted_importance[::-1])):
    ax_agg.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.4f}',
                va='center', color='white', fontsize=9)

ax_agg.set_xlabel('Average Normalized Importance', color='#AABBCC', fontsize=11, fontweight='bold')
ax_agg.set_title('Aggregated Feature Importance\nAverage across all 15 models (normalized scale)',
                  color='white', fontsize=13, fontweight='bold', pad=15)
ax_agg.tick_params(colors='#AABBCC', labelsize=10)
ax_agg.xaxis.grid(True, color='#223344', linestyle='--', alpha=0.5)
ax_agg.set_axisbelow(True)

for spine in ax_agg.spines.values():
    spine.set_visible(False)

fig_agg.tight_layout()
agg_path = f'{OUTPUT_DIR}/feature_importance_aggregated_top_8.png'
plt.savefig(agg_path, dpi=150, bbox_inches='tight', facecolor='#0F1923')
print(f"\nAggregated feature importance chart saved → {agg_path}")
plt.close()

# Save to CSV
feature_avg_df.to_csv(f'{OUTPUT_DIR}/feature_importance_ranking_top_8.csv', index=False)
print(f"Feature ranking saved → {OUTPUT_DIR}/feature_importance_ranking_top_8.csv")



  AGGREGATED FEATURE IMPORTANCE (Average across all models)

      Feature  Avg_Importance
         hour        0.584983
         lag1        0.436445
    dayofyear        0.068025
load_shedding        0.024791
    dayofweek        0.014924
       lag168        0.012948
        month        0.006836
  roll24_mean        0.006288
    temp_mean        0.002671
    wspd_mean        0.000964
        lag24        0.000645
   roll24_std        0.000219

Aggregated feature importance chart saved → ./feature_importance_aggregated_top_8.png
Feature ranking saved → ./feature_importance_ranking_top_8.csv


## 📊 Classification Metrics — Accuracy, Precision, Recall, F1-Score
> Demand classes: **Low** (< 9,066 MW) | **Medium** (9,066–11,382 MW) | **High** (> 11,382 MW)

In [44]:
# ── CLASSIFICATION METRICS FOR ALL 15 MODELS ─────────────────────────────
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.metrics import classification_report

# Convert continuous demand to 3 classes using quantile-based bins
bins   = [0, df[TARGET].quantile(0.33), df[TARGET].quantile(0.66), np.inf]
labels = [0, 1, 2]   # 0 = Low, 1 = Medium, 2 = High demand
df['demand_class'] = pd.cut(df[TARGET], bins=bins, labels=labels).astype(int)

# Create class labels for training and test sets
y_tr_cls = pd.cut(y_tr, bins=bins, labels=labels).astype(int).values
y_te_cls = pd.cut(y_te, bins=bins, labels=labels).astype(int).values

print(f'\nDemand class distribution:')
print(f'Training set:')
print(f'  Low  (class 0): {(y_tr_cls==0).sum():,} samples')
print(f'  Med  (class 1): {(y_tr_cls==1).sum():,} samples')
print(f'  High (class 2): {(y_tr_cls==2).sum():,} samples')
print(f'\nTest set:')
print(f'  Low  (class 0): {(y_te_cls==0).sum():,} samples')
print(f'  Med  (class 1): {(y_te_cls==1).sum():,} samples')
print(f'  High (class 2): {(y_te_cls==2).sum():,} samples')

def to_class(y_cont):
    """Convert continuous predictions to class labels."""
    cls = pd.cut(y_cont, bins=bins, labels=labels)
    cls = cls.astype(float).fillna(1).astype(int)
    return np.clip(cls, 0, 2)

def cls_metrics(name, y_true_cls, y_pred_cls):
    """Compute classification metrics: Accuracy, Precision, Recall, F1-Score"""
    acc  = accuracy_score(y_true_cls, y_pred_cls) * 100
    prec = precision_score(y_true_cls, y_pred_cls, average='weighted', zero_division=0) * 100
    rec  = recall_score(y_true_cls, y_pred_cls, average='weighted', zero_division=0) * 100
    f1   = f1_score(y_true_cls, y_pred_cls, average='weighted', zero_division=0) * 100
    return dict(Model=name,
                Accuracy=round(acc,2), Precision=round(prec,2),
                Recall=round(rec,2), F1_Score=round(f1,2))

print("\n" + "="*80)
print("  CLASSIFICATION METRICS: Computing for all 8 models...")
print("="*80)



Demand class distribution:
Training set:
  Low  (class 0): 12,729 samples
  Med  (class 1): 11,525 samples
  High (class 2): 9,700 samples

Test set:
  Low  (class 0): 1,278 samples
  Med  (class 1): 2,483 samples
  High (class 2): 4,728 samples

  CLASSIFICATION METRICS: Computing for all 8 models...


In [45]:
# ── Compute classification metrics for every model prediction ────────────
cls_results = []

# Collect all predictions from the trained models
all_preds = {
    'ARIMA':               pred_arima,
    'SARIMA':              pred_sarima,
    'Gradient Boosting':   pred_gb,
    'XGBoost':             pred_xgb,
    'TCN':                 pred_tcn,
    'Transformer':         pred_trans,
    'Conformal Prediction': pred_cp,
}

# Compute metrics for each model
for model_name, y_pred_cont in all_preds.items():
    y_pred_cls = to_class(pd.Series(y_pred_cont))
    cls_results.append(cls_metrics(model_name, y_te_cls, y_pred_cls))
    print(f"✓ {model_name}")

# Create results DataFrame
df_cls = pd.DataFrame(cls_results).sort_values('F1_Score', ascending=False)
df_cls.insert(0, 'Rank', range(1, len(df_cls)+1))

# Display results
print('\n' + '='*80)
print('  CLASSIFICATION METRICS — ALL 8 MODELS (Ranked by F1-Score)')
print('='*80)
print(df_cls.to_string(index=False))
print('='*80)

# Save results to CSV
df_cls.to_csv(f'{OUTPUT_DIR}/classification_metrics_top_8.csv', index=False)
print(f'\nResults saved → {OUTPUT_DIR}/classification_metrics_top_8.csv')

# Print top performer
print(f'\n✓ Top Model: {df_cls.iloc[0]["Model"]} with F1-Score of {df_cls.iloc[0]["F1_Score"]:.2f}%')


✓ ARIMA
✓ SARIMA
✓ Gradient Boosting
✓ XGBoost
✓ TCN
✓ Transformer
✓ Conformal Prediction

  CLASSIFICATION METRICS — ALL 8 MODELS (Ranked by F1-Score)
 Rank                Model  Accuracy  Precision  Recall  F1_Score
    1    Gradient Boosting     94.71      94.82   94.71     94.75
    2              XGBoost     94.66      94.80   94.66     94.71
    3                  TCN     94.37      94.41   94.37     94.38
    4          Transformer     94.18      94.32   94.18     94.23
    5 Conformal Prediction     93.82      93.91   93.82     93.85
    6               SARIMA     91.72      91.70   91.72     91.71
    7                ARIMA     91.06      91.07   91.06     91.06

Results saved → ./classification_metrics_top_8.csv

✓ Top Model: Gradient Boosting with F1-Score of 94.75%


In [46]:
# ── VISUALIZATION 1: Classification Metrics Dashboard ──────────────────────
import matplotlib.patches as mpatches
from sklearn.metrics import confusion_matrix

print("\n" + "="*80)
print("  GENERATING CLASSIFICATION METRICS VISUALIZATIONS")
print("="*80)

# Color scheme for model categories
CAT_COLOR = {
    'ARIMA':'#4E79A7','SARIMA':'#4E79A7','Holt-Winters':'#4E79A7','Exp. Smoothing':'#4E79A7',
    'Random Forest':'#F28E2B','Gradient Boosting':'#F28E2B','XGBoost':'#F28E2B','SVR':'#F28E2B',
    'LSTM':'#59A14F','GRU':'#59A14F','TCN':'#59A14F','Transformer':'#59A14F',
    'BSTS':'#B07AA1','Quantile Regression':'#B07AA1','Conformal Prediction':'#B07AA1',
}
BG, AX = '#0F1923', '#151F2B'

# Extract data for plotting
models_s  = df_cls['Model'].tolist()
f1_s      = df_cls['F1_Score'].tolist()
acc_s     = df_cls['Accuracy'].tolist()
prec_s    = df_cls['Precision'].tolist()
rec_s     = df_cls['Recall'].tolist()
col_s     = [CAT_COLOR.get(m,'#888') for m in models_s]

# Create main figure
fig = plt.figure(figsize=(24, 28), facecolor=BG)
fig.patch.set_facecolor(BG)

def style_ax(ax, title, xlabel, ylabel=''):
    """Apply consistent styling to axes"""
    ax.set_facecolor(AX)
    ax.set_title(title, color='white', fontsize=13, fontweight='bold', pad=12)
    ax.set_xlabel(xlabel, color='#AABBCC', fontsize=11)
    if ylabel:
        ax.set_ylabel(ylabel, color='#AABBCC', fontsize=11)
    ax.tick_params(colors='#AABBCC', labelsize=9)
    ax.xaxis.grid(True, color='#223344', linestyle='--', alpha=0.5)
    ax.set_axisbelow(True)
    for sp in ax.spines.values():
        sp.set_visible(False)

# ─────────────────────────────────────────────────────────────────────────────
# Plot 1: F1-Score — Full Width (spanning 2 columns)
# ─────────────────────────────────────────────────────────────────────────────
ax1 = plt.subplot2grid((4, 3), (0, 0), colspan=3, fig=fig)
bars = ax1.barh(models_s[::-1], f1_s[::-1], color=col_s[::-1], edgecolor='none', height=0.65)
for i, bar in enumerate(bars):
    idx = len(models_s) - 1 - i
    val = f1_s[idx]
    # Highlight top performers
    if val >= 94:
        bar.set_edgecolor('gold')
        bar.set_linewidth(1.5)
    ax1.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
             f'{val:.2f}%', va='center', color='white', fontsize=9)
ax1.set_xlim(20, 102)
style_ax(ax1, 'F1-Score — All 15 Models (Ranked Best → Worst)', 'F1-Score (%)')
leg_patches = [
    mpatches.Patch(color='#4E79A7', label='Statistical/Hybrid'),
    mpatches.Patch(color='#F28E2B', label='Machine Learning'),
    mpatches.Patch(color='#59A14F', label='Deep Learning'),
    mpatches.Patch(color='#B07AA1', label='Probabilistic'),
]
ax1.legend(handles=leg_patches, loc='lower right', framealpha=0.25,
           labelcolor='white', facecolor='#223344', fontsize=10)

# ─────────────────────────────────────────────────────────────────────────────
# Plot 2: Accuracy
# ─────────────────────────────────────────────────────────────────────────────
ax2 = plt.subplot2grid((4, 3), (1, 0), fig=fig)
ax2.barh(models_s[::-1], acc_s[::-1], color=col_s[::-1], edgecolor='none', height=0.65)
style_ax(ax2, 'Accuracy (%)', 'Accuracy (%)')
ax2.set_xlim(20, 105)

# ─────────────────────────────────────────────────────────────────────────────
# Plot 3: Precision
# ─────────────────────────────────────────────────────────────────────────────
ax3 = plt.subplot2grid((4, 3), (1, 1), fig=fig)
ax3.barh(models_s[::-1], prec_s[::-1], color=col_s[::-1], edgecolor='none', height=0.65)
style_ax(ax3, 'Precision (%)', 'Precision (%)')
ax3.set_xlim(20, 105)

# ─────────────────────────────────────────────────────────────────────────────
# Plot 4: Recall
# ─────────────────────────────────────────────────────────────────────────────
ax4 = plt.subplot2grid((4, 3), (1, 2), fig=fig)
ax4.barh(models_s[::-1], rec_s[::-1], color=col_s[::-1], edgecolor='none', height=0.65)
style_ax(ax4, 'Recall (%)', 'Recall (%)')
ax4.set_xlim(20, 105)

# ─────────────────────────────────────────────────────────────────────────────
# Plot 5: Grouped Bar Chart — Top 10 Models (All 4 metrics)
# ─────────────────────────────────────────────────────────────────────────────
ax5 = plt.subplot2grid((4, 3), (2, 0), colspan=3, fig=fig)
ax5.set_facecolor(AX)
top10 = df_cls.head(10)
names10 = top10['Model'].tolist()
x = np.arange(len(names10))
w = 0.19

bars1 = ax5.bar(x - 1.5*w, top10['Accuracy'], w, label='Accuracy', color='#4E79A7', edgecolor='white', linewidth=0.5)
bars2 = ax5.bar(x - 0.5*w, top10['Precision'], w, label='Precision', color='#59A14F', edgecolor='white', linewidth=0.5)
bars3 = ax5.bar(x + 0.5*w, top10['Recall'], w, label='Recall', color='#F28E2B', edgecolor='white', linewidth=0.5)
bars4 = ax5.bar(x + 1.5*w, top10['F1_Score'], w, label='F1-Score', color='#B07AA1', edgecolor='white', linewidth=0.5)

ax5.set_xticks(x)
ax5.set_xticklabels(names10, rotation=45, ha='right', color='#AABBCC', fontsize=9)
ax5.tick_params(colors='#AABBCC', labelsize=9)
ax5.set_ylim(85, 102)
ax5.yaxis.grid(True, color='#223344', linestyle='--', alpha=0.5)
ax5.set_axisbelow(True)
for sp in ax5.spines.values():
    sp.set_visible(False)
ax5.set_title('Top 10 Models — Accuracy, Precision, Recall & F1-Score', 
              color='white', fontsize=13, fontweight='bold', pad=12)
ax5.set_ylabel('Score (%)', color='#AABBCC', fontsize=11)
ax5.legend(framealpha=0.25, labelcolor='white', facecolor='#223344', fontsize=10, loc='lower left')

# ─────────────────────────────────────────────────────────────────────────────
# Plot 6: Confusion Matrix for Top 3 Models
# ─────────────────────────────────────────────────────────────────────────────
top_models = df_cls.head(3)['Model'].tolist()

for idx, model_name in enumerate(top_models):
    ax_cm = plt.subplot2grid((4, 3), (3, idx), fig=fig)
    ax_cm.set_facecolor(AX)
    
    # Get predictions for this model
    y_pred_cont = all_preds[model_name]
    y_pred_cls = to_class(pd.Series(y_pred_cont))
    
    # Compute confusion matrix
    cm = confusion_matrix(y_te_cls, y_pred_cls)
    
    # Normalize for better visualization
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Create heatmap
    im = ax_cm.imshow(cm_normalized, interpolation='nearest', cmap='YlOrRd', vmin=0, vmax=1)
    
    # Add text annotations
    class_names = ['Low', 'Medium', 'High']
    tick_marks = np.arange(3)
    ax_cm.set_xticks(tick_marks)
    ax_cm.set_yticks(tick_marks)
    ax_cm.set_xticklabels(class_names, color='#AABBCC', fontsize=9)
    ax_cm.set_yticklabels(class_names, color='#AABBCC', fontsize=9)
    
    # Add count annotations
    for i in range(3):
        for j in range(3):
            text = ax_cm.text(j, i, f'{cm[i, j]}\n({cm_normalized[i, j]:.1%})',
                             ha="center", va="center", color="white" if cm_normalized[i, j] > 0.5 else "black",
                             fontsize=9, fontweight='bold')
    
    ax_cm.set_xlabel('Predicted', color='#AABBCC', fontsize=10)
    ax_cm.set_ylabel('Actual', color='#AABBCC', fontsize=10)
    ax_cm.set_title(f'{model_name}\n(F1: {df_cls[df_cls["Model"]==model_name]["F1_Score"].values[0]:.2f}%)',
                   color='white', fontsize=11, fontweight='bold', pad=10)

fig.suptitle('Electricity Demand Forecasting — Classification Metrics Dashboard\n'
             'Accuracy · Precision · Recall · F1-Score | 3-Class Problem: Low / Medium / High Demand',
             color='white', fontsize=16, fontweight='bold', y=0.995)
fig.subplots_adjust(top=0.97, hspace=0.45, wspace=0.35, left=0.08, right=0.98, bottom=0.03)

chart_path = f'{OUTPUT_DIR}/classification_metrics_dashboard_top_8.png'
plt.savefig(chart_path, dpi=150, bbox_inches='tight', facecolor=BG)
print(f'\n✓ Classification dashboard saved → {chart_path}')
plt.close()

print("="*80)



  GENERATING CLASSIFICATION METRICS VISUALIZATIONS

✓ Classification dashboard saved → ./classification_metrics_dashboard_top_8.png


In [49]:
# ── SAVE GRADIENT BOOSTING MODEL AND SCALER ───────────────────────────────
import pickle
import json
from pathlib import Path

# Create models directory
MODELS_DIR = Path('./models')
MODELS_DIR.mkdir(exist_ok=True)

print("="*80)
print("  SAVING TRAINED MODELS FOR STREAMLIT DEPLOYMENT")
print("="*80)

# ── 1. SAVE GRADIENT BOOSTING MODEL ───────────────────────────────────────
model_path = MODELS_DIR / 'gradient_boosting_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(m_gb, f)
print(f"✓ Gradient Boosting model saved → {model_path}")

# ── 2. SAVE STANDARD SCALER ──────────────────────────────────────────────
scaler_path = MODELS_DIR / 'standard_scaler.pkl'
with open(scaler_path, 'wb') as f:
    pickle.dump(sc, f)
print(f"✓ Standard Scaler saved → {scaler_path}")

# ── 3. SAVE FEATURE NAMES (as JSON for easy reading) ────────────────────
features_path = MODELS_DIR / 'feature_names.json'
with open(features_path, 'w') as f:
    json.dump(FEATURES, f, indent=2)
print(f"✓ Feature names saved → {features_path}")

# ── 4. SAVE MODEL METADATA ───────────────────────────────────────────────
metadata = {
    'model_type': 'GradientBoostingRegressor',
    'n_estimators': 200,
    'learning_rate': 0.05,
    'max_depth': 5,
    'random_state': 42,
    'features': FEATURES,
    'n_features': len(FEATURES),
    'trained_samples': len(X_train),
    'test_samples': len(X_test),
}

metadata_path = MODELS_DIR / 'model_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✓ Model metadata saved → {metadata_path}")

# ── 5. ALSO SAVE ALL OTHER MODELS (Optional) ─────────────────────────────
models_to_save = {
    'xgboost': m_xgb,
    'transformer': m_trans,
    'tcn': m_tcn,
    'conformal_prediction': m_cp,
    'arima': m_arima,
    'sarima': m_sarima,
}

for model_name, model_obj in models_to_save.items():
    model_file = MODELS_DIR / f'{model_name}_model.pkl'
    with open(model_file, 'wb') as f:
        pickle.dump(model_obj, f)
    print(f"✓ {model_name.upper()} model saved → {model_file}")

print("="*80)
print(f"\n✅ All models saved to: {MODELS_DIR.absolute()}")
print(f"\nFiles created:")
print(f"  📦 gradient_boosting_model.pkl    (Main model)")
print(f"  📦 standard_scaler.pkl            (Data preprocessor)")
print(f"  📦 feature_names.json             (Feature list)")
print(f"  📦 model_metadata.json            (Model config)")
print(f"  📦 xgboost_model.pkl              (Alternative model)")
print(f"  📦 transformer_model.pkl          (Alternative model)")
print(f"  📦 tcn_model.pkl                  (Alternative model)")
print(f"  📦 conformal_prediction_model.pkl (Alternative model)")
print("="*80)

  SAVING TRAINED MODELS FOR STREAMLIT DEPLOYMENT
✓ Gradient Boosting model saved → models\gradient_boosting_model.pkl
✓ Standard Scaler saved → models\standard_scaler.pkl
✓ Feature names saved → models\feature_names.json
✓ Model metadata saved → models\model_metadata.json
✓ XGBOOST model saved → models\xgboost_model.pkl
✓ TRANSFORMER model saved → models\transformer_model.pkl
✓ TCN model saved → models\tcn_model.pkl
✓ CONFORMAL_PREDICTION model saved → models\conformal_prediction_model.pkl
✓ ARIMA model saved → models\arima_model.pkl
✓ SARIMA model saved → models\sarima_model.pkl

✅ All models saved to: d:\Users\Md Mahfuzur Rahman\Desktop\Research\For me\models

Files created:
  📦 gradient_boosting_model.pkl    (Main model)
  📦 standard_scaler.pkl            (Data preprocessor)
  📦 feature_names.json             (Feature list)
  📦 model_metadata.json            (Model config)
  📦 xgboost_model.pkl              (Alternative model)
  📦 transformer_model.pkl          (Alternative model)
 

In [10]:
# ── CALCULATE 24-HOUR MEAN DEMAND (Last 1 Year) ──────────────────────────
import json
print("="*80)
print("  CALCULATING 24-HOUR MEAN DEMAND FROM LAST 1 YEAR")
print("="*80)

# Create models directory if it doesn't exist
from pathlib import Path
MODELS_DIR = Path('./models')
MODELS_DIR.mkdir(exist_ok=True)

# Get last 365 days of data
last_year_data = df.tail(365*24)  # Assuming hourly data

# Create a copy for processing
df_temp = last_year_data.copy()

# Extract hour from datetime if not already present
if 'hour' not in df_temp.columns:
    df_temp['hour'] = pd.to_datetime(df_temp['datetime']).dt.hour

# Calculate mean demand for each hour across the last year
hourly_mean_demand = df_temp.groupby('hour')['demand_mw'].mean().round(2)

# Convert to dictionary: {hour: mean_demand}
hourly_mean_dict = {int(hour): float(demand) for hour, demand in hourly_mean_demand.items()}

# Ensure all 24 hours are present (fill missing with overall mean if needed)
overall_mean = df_temp['demand_mw'].mean()
for hour in range(24):
    if hour not in hourly_mean_dict:
        hourly_mean_dict[hour] = round(overall_mean, 2)

# Sort by hour
hourly_mean_dict = {k: hourly_mean_dict[k] for k in sorted(hourly_mean_dict.keys())}

# Display the hourly means
print("\n24-Hour Mean Demand Values (Last 1 Year):")
print("="*80)
for hour in range(24):
    print(f"  Hour {hour:2d}:00 → {hourly_mean_dict[hour]:8.1f} MW")
print("="*80)

# Save to JSON for Streamlit
hourly_means_path = MODELS_DIR / 'hourly_mean_demand.json'
with open(hourly_means_path, 'w') as f:
    json.dump(hourly_mean_dict, f, indent=2)
print(f"\n✓ Hourly mean demand saved → {hourly_means_path}")


  CALCULATING 24-HOUR MEAN DEMAND FROM LAST 1 YEAR

24-Hour Mean Demand Values (Last 1 Year):
  Hour  0:00 →  11740.6 MW
  Hour  1:00 →  11905.3 MW
  Hour  2:00 →  11467.3 MW
  Hour  3:00 →  11146.0 MW
  Hour  4:00 →  10858.1 MW
  Hour  5:00 →  10602.2 MW
  Hour  6:00 →  10388.0 MW
  Hour  7:00 →  10610.7 MW
  Hour  8:00 →  10891.6 MW
  Hour  9:00 →  11093.0 MW
  Hour 10:00 →  11196.8 MW
  Hour 11:00 →  11465.7 MW
  Hour 12:00 →  11949.0 MW
  Hour 13:00 →  12316.0 MW
  Hour 14:00 →  12069.5 MW
  Hour 15:00 →  11974.6 MW
  Hour 16:00 →  11750.9 MW
  Hour 17:00 →  11679.8 MW
  Hour 18:00 →  12106.6 MW
  Hour 19:00 →  12926.9 MW
  Hour 20:00 →  13109.9 MW
  Hour 21:00 →  13088.2 MW
  Hour 22:00 →  12844.4 MW
  Hour 23:00 →  12610.5 MW

✓ Hourly mean demand saved → models\hourly_mean_demand.json


In [11]:
hourly_mean_demand

hour
1     11905.26
2     11467.28
3     11146.04
4     10858.11
5     10602.22
6     10388.01
7     10610.71
8     10891.60
9     11093.05
10    11196.77
11    11465.69
12    11948.95
13    12316.01
14    12069.55
15    11974.60
16    11750.90
17    11679.75
18    12106.63
19    12926.85
20    13109.89
21    13088.19
22    12844.43
23    12610.48
Name: demand_mw, dtype: float64